# Function Calling avanzado con Claude

Tool use nativo de Anthropic: herramientas paralelas, schemas complejos,
validación con Pydantic y patrones de agente de producción.

In [ ]:
import anthropic, os, json, time
from pydantic import BaseModel, Field, field_validator
from typing import Literal, Annotated

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('Cliente listo')

## 1. Herramienta básica y el bucle tool_use

In [ ]:
HERRAMIENTAS = [
    {
        'name': 'calcular',
        'description': 'Evalúa expresiones matemáticas de forma segura.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'expresion': {
                    'type': 'string',
                    'description': 'Expresión matemática, ej: "2 ** 10 + 500 * 3"'
                }
            },
            'required': ['expresion']
        }
    }
]

import ast, operator

OPERADORES = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.Mod: operator.mod,
}

def calcular_seguro(expresion: str) -> float:
    def evaluar(node):
        if isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            op = OPERADORES.get(type(node.op))
            if op is None:
                raise ValueError(f'Operador no permitido')
            return op(evaluar(node.left), evaluar(node.right))
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -evaluar(node.operand)
        raise ValueError(f'Expresión no permitida: {ast.dump(node)}')
    return evaluar(ast.parse(expresion, mode='eval').body)

def ejecutar_herramienta(nombre: str, args: dict) -> str:
    if nombre == 'calcular':
        try:
            resultado = calcular_seguro(args['expresion'])
            return json.dumps({'resultado': resultado, 'expresion': args['expresion']})
        except Exception as e:
            return json.dumps({'error': str(e)})
    return json.dumps({'error': 'Herramienta desconocida'})

def agente_con_herramientas(pregunta: str, herramientas: list) -> str:
    mensajes = [{'role': 'user', 'content': pregunta}]

    while True:
        resp = client.messages.create(
            model=MODELO, max_tokens=500,
            tools=herramientas,
            messages=mensajes,
        )

        if resp.stop_reason == 'end_turn':
            return next(b.text for b in resp.content if hasattr(b, 'text'))

        if resp.stop_reason == 'tool_use':
            mensajes.append({'role': 'assistant', 'content': resp.content})
            resultados_tools = []
            for bloque in resp.content:
                if bloque.type == 'tool_use':
                    print(f'  → {bloque.name}({bloque.input})')
                    resultado = ejecutar_herramienta(bloque.name, bloque.input)
                    print(f'  ← {resultado}')
                    resultados_tools.append({
                        'type': 'tool_result',
                        'tool_use_id': bloque.id,
                        'content': resultado,
                    })
            mensajes.append({'role': 'user', 'content': resultados_tools})

respuesta = agente_con_herramientas(
    '¿Cuánto es 2 elevado a 16? Y también calcula el 15% de 7850.',
    HERRAMIENTAS
)
print('\nRespuesta:', respuesta)

## 2. Múltiples herramientas y tool_choice

In [ ]:
HERRAMIENTAS_EMPRESA = [
    {
        'name': 'buscar_cliente',
        'description': 'Busca un cliente por nombre o email en la base de datos.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Nombre o email del cliente'},
                'campo': {'type': 'string', 'enum': ['nombre', 'email', 'id'], 'default': 'nombre'},
            },
            'required': ['query'],
        },
    },
    {
        'name': 'obtener_pedidos',
        'description': 'Obtiene los pedidos de un cliente por su ID.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'cliente_id': {'type': 'integer'},
                'estado': {'type': 'string', 'enum': ['todos', 'pendiente', 'enviado', 'entregado'], 'default': 'todos'},
                'limite': {'type': 'integer', 'default': 10, 'minimum': 1, 'maximum': 100},
            },
            'required': ['cliente_id'],
        },
    },
    {
        'name': 'crear_ticket_soporte',
        'description': 'Crea un ticket de soporte para un cliente.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'cliente_id': {'type': 'integer'},
                'asunto': {'type': 'string'},
                'descripcion': {'type': 'string'},
                'prioridad': {'type': 'string', 'enum': ['baja', 'media', 'alta', 'critica'], 'default': 'media'},
            },
            'required': ['cliente_id', 'asunto', 'descripcion'],
        },
    },
]

# Datos de prueba
DB_CLIENTES = {
    'Ana García': {'id': 101, 'email': 'ana@empresa.com', 'plan': 'pro'},
    'Carlos López': {'id': 102, 'email': 'carlos@corp.es', 'plan': 'enterprise'},
}
DB_PEDIDOS = {
    101: [{'id': 'P-001', 'estado': 'entregado', 'total': 249.99}, {'id': 'P-002', 'estado': 'pendiente', 'total': 89.00}],
    102: [{'id': 'P-003', 'estado': 'enviado', 'total': 1200.00}],
}

def ejecutar_herramienta_empresa(nombre: str, args: dict) -> str:
    if nombre == 'buscar_cliente':
        for nombre_c, datos in DB_CLIENTES.items():
            if args['query'].lower() in nombre_c.lower() or args['query'] == datos.get('email'):
                return json.dumps({'encontrado': True, 'cliente': {'nombre': nombre_c, **datos}})
        return json.dumps({'encontrado': False})
    elif nombre == 'obtener_pedidos':
        pedidos = DB_PEDIDOS.get(args['cliente_id'], [])
        if args.get('estado', 'todos') != 'todos':
            pedidos = [p for p in pedidos if p['estado'] == args['estado']]
        return json.dumps({'pedidos': pedidos[:args.get('limite', 10)], 'total': len(pedidos)})
    elif nombre == 'crear_ticket_soporte':
        ticket_id = f'TK-{int(time.time()) % 10000:04d}'
        return json.dumps({'creado': True, 'ticket_id': ticket_id, **args})
    return json.dumps({'error': 'Herramienta desconocida'})

def agente_empresa(pregunta: str) -> str:
    mensajes = [{'role': 'user', 'content': pregunta}]
    system = 'Eres el asistente de CRM de la empresa. Usa las herramientas para responder con datos reales.'

    while True:
        resp = client.messages.create(
            model=MODELO, max_tokens=600, system=system,
            tools=HERRAMIENTAS_EMPRESA, messages=mensajes,
        )
        if resp.stop_reason == 'end_turn':
            return next((b.text for b in resp.content if hasattr(b, 'text')), '')
        mensajes.append({'role': 'assistant', 'content': resp.content})
        resultados = []
        for b in resp.content:
            if b.type == 'tool_use':
                print(f'  🔧 {b.name}({json.dumps(b.input, ensure_ascii=False)})')
                resultado = ejecutar_herramienta_empresa(b.name, b.input)
                resultados.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        mensajes.append({'role': 'user', 'content': resultados})

print('Consulta: ¿Qué pedidos tiene Ana García pendientes? Si hay alguno, crea un ticket de seguimiento.')
print()
respuesta = agente_empresa('¿Qué pedidos tiene Ana García pendientes? Si hay alguno, crea un ticket de seguimiento con prioridad alta.')
print(f'\nRespuesta: {respuesta}')

## 3. Validación de inputs con Pydantic

In [ ]:
from pydantic import BaseModel, Field, field_validator

class InputBuscarCliente(BaseModel):
    query: str = Field(min_length=2, max_length=100)
    campo: Literal['nombre', 'email', 'id'] = 'nombre'

class InputCrearTicket(BaseModel):
    cliente_id: int = Field(gt=0)
    asunto: str = Field(min_length=5, max_length=200)
    descripcion: str = Field(min_length=10)
    prioridad: Literal['baja', 'media', 'alta', 'critica'] = 'media'

    @field_validator('asunto')
    @classmethod
    def asunto_no_spam(cls, v: str) -> str:
        palabras_spam = ['urgente urgente', 'ayuda!!!', '¡¡¡']
        if any(p in v.lower() for p in palabras_spam):
            raise ValueError('El asunto parece spam')
        return v.strip()

VALIDADORES = {
    'buscar_cliente': InputBuscarCliente,
    'crear_ticket_soporte': InputCrearTicket,
}

def ejecutar_con_validacion(nombre: str, args: dict) -> str:
    validador = VALIDADORES.get(nombre)
    if validador:
        try:
            args_validados = validador(**args).model_dump()
        except Exception as e:
            return json.dumps({'error': f'Datos inválidos: {str(e)}'})
    else:
        args_validados = args
    return ejecutar_herramienta_empresa(nombre, args_validados)

# Probar validación
casos_test = [
    ('buscar_cliente', {'query': 'A', 'campo': 'nombre'}),         # query muy corta
    ('buscar_cliente', {'query': 'Ana García', 'campo': 'nombre'}),  # válida
    ('crear_ticket_soporte', {'cliente_id': -1, 'asunto': 'Bug', 'descripcion': 'x'}),  # inválida
    ('crear_ticket_soporte', {'cliente_id': 101, 'asunto': 'Bug en login', 'descripcion': 'El botón de login no responde en mobile.', 'prioridad': 'alta'}),  # válida
]

for nombre, args in casos_test:
    resultado = ejecutar_con_validacion(nombre, args)
    print(f'{nombre}({args})')
    print(f'  → {resultado[:100]}\n')

## 4. tool_choice — forzar o desactivar herramientas

In [ ]:
# tool_choice controla cuándo usa herramientas el modelo

pregunta = '¿Cuáles son los pedidos del cliente 101?'

for modo in ['auto', 'any', {'type': 'tool', 'name': 'obtener_pedidos'}]:
    resp = client.messages.create(
        model=MODELO, max_tokens=200,
        tools=HERRAMIENTAS_EMPRESA,
        tool_choice=modo if isinstance(modo, dict) else {'type': modo},
        messages=[{'role': 'user', 'content': pregunta}],
    )
    modo_str = modo if isinstance(modo, str) else f'force:{modo["name"]}'
    herramienta_usada = next(
        (b.name for b in resp.content if hasattr(b, 'name')), 'ninguna'
    )
    print(f'tool_choice={modo_str}: stop_reason={resp.stop_reason}, herramienta={herramienta_usada}')